In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)


TensorFlow Version: 2.19.0


In [ ]:
IMG_HEIGHT = 224     # EfficientNet best size
IMG_WIDTH  = 224
CHANNELS   = 3
NUM_CLASSES = 7
BATCH_SIZE = 16      # small batch for stability
EPOCHS_HEAD = 15     # classifier training
EPOCHS_FINE = 20     # fine-tuning


In [ ]:
TRAIN_DIR = "/content/drive/MyDrive/Colab Notebooks/Indian Currency dataset/train"
VAL_DIR   = "/content/drive/MyDrive/Colab Notebooks/Indian Currency dataset/valid"
TEST_DIR  = "/content/drive/MyDrive/Colab Notebooks/Indian Currency dataset/test"


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.15,
    brightness_range=[0.9, 1.1],
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode="nearest"
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="sparse"
)

val_data = val_gen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

test_data = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)


Found 3173 images belonging to 7 classes.
Found 315 images belonging to 7 classes.
Found 83 images belonging to 7 classes.


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

base_model = EfficientNetB0(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
    include_top=False,
    weights="imagenet"
)

# 🔒 Stage 1: Freeze entire backbone
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,219,562 (16.10 MB)

 Trainable params: 167,431 (654.03 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]


In [ ]:
print("🚀 Stage 1: Training classifier head")

history_head = model.fit(
    train_data,
    epochs=EPOCHS_HEAD,
    validation_data=val_data,
    callbacks=callbacks
)


🚀 Stage 1: Training classifier head


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 1182s 6s/step - accuracy: 0.4395 - loss: 1.7699 - val_accuracy: 0.8540 - val_loss: 0.5444 - learning_rate: 0.0010
Epoch 2/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.7446 - loss: 0.7443 - val_accuracy: 0.8794 - val_loss: 0.3712 - learning_rate: 0.0010
Epoch 3/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 385s 2s/step - accuracy: 0.8304 - loss: 0.5397 - val_accuracy: 0.9111 - val_loss: 0.2981 - learning_rate: 0.0010
Epoch 4/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 391s 2s/step - accuracy: 0.8336 - loss: 0.4808 - val_accuracy: 0.9079 - val_loss: 0.3031 - learning_rate: 0.0010
Epoch 5/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 386s 2s/step - accuracy: 0.8613 - loss: 0.3956 - val_accuracy: 0.9397 - val_loss: 0.2241 - learning_rate: 0.0010
Epoch 6/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 374s 2s/step - accuracy: 0.8753 - loss: 0.3610 - val_accuracy: 0.9333 - val_loss: 0.2167 - learning_rate: 0.0010
Epoch 7/15
199/199 ━━━━━━━━━━━━━━━━━━━━ 381s 2s/step - accuracy: 0.9001 - loss: 0

In [ ]:
print("🔧 Stage 2: Fine-tuning EfficientNet")

base_model.trainable = True

# Freeze most layers (small dataset)
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Freeze ALL BatchNorm layers
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history_fine = model.fit(
    train_data,
    epochs=EPOCHS_FINE,
    validation_data=val_data,
    callbacks=callbacks
)


🔧 Stage 2: Fine-tuning EfficientNet


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,219,562 (16.10 MB)

 Trainable params: 1,650,791 (6.30 MB)

 Non-trainable params: 2,568,771 (9.80 MB)

Epoch 1/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 447s 2s/step - accuracy: 0.9197 - loss: 0.2312 - val_accuracy: 0.9683 - val_loss: 0.1770 - learning_rate: 1.0000e-05
Epoch 2/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 424s 2s/step - accuracy: 0.9296 - loss: 0.2071 - val_accuracy: 0.9714 - val_loss: 0.1667 - learning_rate: 1.0000e-05
Epoch 3/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 434s 2s/step - accuracy: 0.9297 - loss: 0.2029 - val_accuracy: 0.9683 - val_loss: 0.1577 - learning_rate: 1.0000e-05
Epoch 4/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 430s 2s/step - accuracy: 0.9427 - loss: 0.1652 - val_accuracy: 0.9651 - val_loss: 0.1591 - learning_rate: 1.0000e-05
Epoch 5/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 428s 2s/step - accuracy: 0.9469 - loss: 0.1720 - val_accuracy: 0.9651 - val_loss: 0.1553 - learning_rate: 1.0000e-05
Epoch 6/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 430s 2s/step - accuracy: 0.9470 - loss: 0.1545 - val_accuracy: 0.9746 - val_loss: 0.1471 - learning_rate: 1.0000e-05
Epoch 7/20
199/199 ━━━━━━━━━━━━━━━━━━━━ 414s 2s/step - acc

In [ ]:
print("📊 Evaluating on Test Set")

test_loss, test_acc = model.evaluate(test_data)
print(f"Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# IMPORTANT: reset generator
test_data.reset()

# Predict probabilities
y_pred_probs = model.predict(test_data, verbose=1)

# Convert probabilities → class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Ground truth labels
y_true = test_data.classes

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")

print("\n📊 Model Performance Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

class_names = list(test_data.class_indices.keys())

print("\n📋 Detailed Classification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))


In [ ]:
# Export model (for local / deployment use)
model.export("/content/drive/MyDrive/efficientnet_currency_detection")

# Save class names
import json

class_names = [None] * len(train_data.class_indices)
for name, index in train_data.class_indices.items():
    class_names[index] = name

with open("/content/drive/MyDrive/class_names.json", "w") as f:
    json.dump(class_names, f)

print("✅ Model and class names saved")
